# RQ3: Defense analysis

Requires the outputs of `bash_scripts/rq3_defense.sh` (and optionally `rq3_finbert_baseline.ipynb`) in `outputs/bias_defense/`.

1. Accuracy per condition, added to each model's classification report
2. Accuracy on pairwise-bias sentences per condition
3. Accuracy per single bias type per condition
4. Format failures of chain-of-thought outputs

In [ ]:
import os

import pandas as pd
from sklearn.metrics import accuracy_score

DEFENSE_DIR = "../outputs/bias_defense"
ANALYSIS_DIR = "../outputs/analysis"
os.makedirs(ANALYSIS_DIR, exist_ok=True)

MODELS = ["finbert-tone", "gpt-5.4-mini", "claude-haiku-4.5", "gemini-3-flash", "llama-3.1-8b", "llama-3.1-70b",
          "llama-3.3-70b", "qwen3-8b", "qwen3.5-9b", "qwen3-32b", "gemma-4-12b", "gemma-4-31b"]
CONDITIONS = ["2_no_defense", "3_generic_defense", "4_cot", "4_cot_clean", "5_targeted_detected",
              "6_targeted_gold", "7_rewrite_detected", "8_rewrite_gold"]
BIAS_TYPES = ["anchoring", "authority", "framing", "herding", "recency", "overconfidence"]

def predictions_path(model):
    return os.path.join(DEFENSE_DIR, model, f"predictions_{model}.csv")

## Accuracy per condition

In [ ]:
for model_dir in sorted(os.listdir(DEFENSE_DIR)):
    model_path = os.path.join(DEFENSE_DIR, model_dir)
    if not os.path.isdir(model_path):
        continue
    
    predictions_file = os.path.join(model_path, f"predictions_{model_dir}.csv")
    report_file = os.path.join(model_path, f"classification_report_{model_dir}.csv")
    
    if not os.path.exists(predictions_file) or not os.path.exists(report_file):
        print(f"Skipping {model_dir}: missing files")
        continue
    
    predictions_df = pd.read_csv(predictions_file)
    report_df = pd.read_csv(report_file, index_col=0)
    
    # Count NaN parsed_sentiment
    total_rows = len(predictions_df)
    nan_count = predictions_df["parsed_sentiment"].isna().sum()
    nan_pct = (nan_count / total_rows) * 100
    
    # Fill NaN with a dummy value that will never match any label
    predictions_df["parsed_sentiment"] = predictions_df["parsed_sentiment"].fillna("__INVALID__")
    
    # Compute accuracy per condition
    accuracy_map = {}
    for condition in predictions_df["condition"].unique():
        cond_df = predictions_df[predictions_df["condition"] == condition]
        acc = accuracy_score(cond_df["label"], cond_df["parsed_sentiment"])
        accuracy_map[condition] = round(acc, 6)
    
    # Add accuracy column to report
    report_df["accuracy"] = report_df["condition"].map(accuracy_map)
    
    # Save as new file
    output_path = os.path.join(model_path, f"classification_report_with_accuracy_{model_dir}.csv")
    report_df.to_csv(output_path)
    print(f"Saved: {output_path}")
    print(f"  NaN predictions: {nan_count}/{total_rows} ({nan_pct:.2f}%)")

## Accuracy on pairwise-bias sentences

In [ ]:
results = {cond: {model: "-" for model in MODELS} for cond in CONDITIONS}

for model in MODELS:
    if not os.path.exists(predictions_path(model)):
        print(f"Skipping {model}: no predictions file")
        continue
    
    df = pd.read_csv(predictions_path(model))
    df_pairwise = df[df["biases"].str.contains(",", na=False)].copy()
    
    # Fill NaN with invalid value to count as incorrect
    df_pairwise["parsed_sentiment"] = df_pairwise["parsed_sentiment"].fillna("__INVALID__")
    
    for condition in CONDITIONS:
        cond_df = df_pairwise[df_pairwise["condition"] == condition]
        
        if len(cond_df) > 0:
            acc = accuracy_score(cond_df["label"], cond_df["parsed_sentiment"])
            results[condition][model] = round(acc, 3)

results_df = pd.DataFrame(results).T
results_df = results_df[MODELS]
results_df.index.name = "condition"

output_path = os.path.join(ANALYSIS_DIR, "defense_pairwise_accuracy.csv")
results_df.to_csv(output_path)
print(f"Saved to: {output_path}")
results_df

## Accuracy per single bias type

In [ ]:
results = []

for model in MODELS:
    if not os.path.exists(predictions_path(model)):
        for bias in BIAS_TYPES:
            for condition in CONDITIONS:
                results.append({"bias_type": bias, "condition": condition, "model": model, "accuracy": "-"})
        continue
    
    df = pd.read_csv(predictions_path(model))
    
    # Fill NaN with invalid value to count as incorrect
    df["parsed_sentiment"] = df["parsed_sentiment"].fillna("__INVALID__")
    
    # Filter to single-bias only (no comma in biases column)
    df_single = df[~df["biases"].str.contains(",", na=False)].copy()
    df_single = df_single[df_single["biases"] != "none"].copy()
    
    for bias in BIAS_TYPES:
        df_bias = df_single[df_single["biases"] == bias]
        
        for condition in CONDITIONS:
            cond_df = df_bias[df_bias["condition"] == condition]
            
            if len(cond_df) > 0:
                acc = accuracy_score(cond_df["label"], cond_df["parsed_sentiment"])
                results.append({"bias_type": bias, "condition": condition, "model": model, "accuracy": round(acc, 3)})
            else:
                results.append({"bias_type": bias, "condition": condition, "model": model, "accuracy": "-"})

results_df = pd.DataFrame(results)
pivot_df = results_df.pivot(index=["bias_type", "condition"], columns="model", values="accuracy")
pivot_df = pivot_df[MODELS]

output_path = os.path.join(ANALYSIS_DIR, "defense_single_bias_accuracy.csv")
pivot_df.to_csv(output_path)
print(f"Saved to: {output_path}")
pivot_df

## Chain-of-thought format failures

A CoT output counts as a format failure when no sentiment label can be found in it.

In [ ]:
def check_format_failure(text: str) -> bool:
    """Check if CoT output has format failure (no valid sentiment found)."""
    text = str(text).lower().strip()
    
    # Check for FINAL ANSWER format first
    if "final answer:" in text:
        answer_part = text.split("final answer:")[-1].strip()
        if any(s in answer_part for s in ["positive", "negative", "neutral"]):
            return False  # Valid format
    
    # Fallback: check if any sentiment word exists anywhere
    if any(s in text for s in ["positive", "negative", "neutral"]):
        return False  # Valid format
    
    return True  # Format failure

results = []

for model in MODELS:
    if not os.path.exists(predictions_path(model)):
        continue
    
    df = pd.read_csv(predictions_path(model))
    
    for condition in ['4_cot', '4_cot_clean']:
        cond_df = df[df['condition'] == condition]
        
        if len(cond_df) == 0:
            continue
        
        failures = cond_df['predicted_sentiment'].apply(check_format_failure).sum()
        total = len(cond_df)
        failure_rate = (failures / total) * 100
        
        results.append({
            'model': model,
            'condition': condition,
            'total': total,
            'format_failures': failures,
            'failure_rate_pct': round(failure_rate, 2)
        })

summary_df = pd.DataFrame(results)

# Pivot for cleaner view
pivot_df = summary_df.pivot(index='model', columns='condition', values=['total', 'format_failures', 'failure_rate_pct'])
pivot_df.columns = [f'{col[1]}_{col[0]}' for col in pivot_df.columns]
pivot_df = pivot_df.reset_index()

output_path = os.path.join(ANALYSIS_DIR, "cot_format_failures.csv")
pivot_df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")
pivot_df